[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.2_vram_budgeting/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue?logo=data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSIyNCIgaGVpZ2h0PSIyNCI+PC9zdmc+)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.2_vram_budgeting/lab.ipynb)

# Lab 1.2: VRAM Budgeting

This lab builds the complete VRAM budget equation: **Total = Weights + KV Cache + Activations + Overhead**.
You will compute each component from first principles and predict whether a model fits on a given GPU.

In [ ]:
import math

# GPU catalog: name -> VRAM in GB
GPU_CATALOG = {
    "T4": 16,
    "A10G": 24,
    "L4": 24,
    "A100-40": 40,
    "A100-80": 80,
    "H100": 80,
    "RTX_Pro_6000": 96,
}

# Bytes per parameter for each precision
BYTES_PER_PARAM = {"FP16": 2, "INT8": 1, "INT4": 0.5}

def weight_memory_gb(num_params_B, precision):
    """Weight memory in GB given params in billions."""
    return num_params_B * 1e9 * BYTES_PER_PARAM[precision] / (1024**3)

print("Setup complete. GPU_CATALOG and helpers loaded.")

## Exercise 1: Weight Memory

Compute weight memory for Llama 8B, 70B, and 405B at FP16, INT8, and INT4.

**Formula:** `Weight Memory (GB) = num_params × bytes_per_param / 1024³`

In [ ]:
models = {"Llama-8B": 8, "Llama-70B": 70, "Llama-405B": 405}
precisions = ["FP16", "INT8", "INT4"]

print(f"{'Model':<12} {'FP16 (GB)':>10} {'INT8 (GB)':>10} {'INT4 (GB)':>10}")
print("-" * 44)
for name, params in models.items():
    row = [weight_memory_gb(params, p) for p in precisions]
    print(f"{name:<12} {row[0]:>10.1f} {row[1]:>10.1f} {row[2]:>10.1f}")

# Key insight: INT4 cuts memory 4x vs FP16, making 70B fit on a single 80GB GPU.

## Exercise 2: KV Cache Size

**Formula:** `KV Cache (GB) = 2 × num_layers × num_kv_heads × head_dim × seq_len × batch_size × bytes / 1024³`

The factor of 2 accounts for both K and V tensors stored per layer.

In [ ]:
def kv_cache_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size, precision="FP16"):
    """KV cache memory in GB."""
    bytes_per_val = BYTES_PER_PARAM[precision]
    return 2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size * bytes_per_val / (1024**3)

# Llama 3.1 8B: 32 layers, 8 KV heads (GQA), head_dim=128
llama8b = {"num_layers": 32, "num_kv_heads": 8, "head_dim": 128}

print("KV Cache for Llama-8B (batch=1, FP16):")
print(f"{'Context Length':<16} {'KV Cache (GB)':>14}")
print("-" * 32)
for ctx in [2048, 8192, 32768, 131072]:
    mem = kv_cache_gb(**llama8b, seq_len=ctx, batch_size=1)
    print(f"{ctx:<16} {mem:>14.3f}")

print("\n# Notice: KV cache grows linearly with context. At 128K tokens it rivals weight memory.")

## Exercise 3: Full VRAM Budget

**Total VRAM = Weights + KV Cache + Activations + Overhead**

- Activations ≈ 5-10% of weight memory (varies by batch size)
- Overhead (CUDA kernels, framework) ≈ 500MB-1.5GB fixed

In [ ]:
def vram_budget_gb(num_params_B, precision, num_layers, num_kv_heads, head_dim,
                   seq_len, batch_size, activation_pct=0.08, overhead_gb=1.0):
    """Full VRAM budget in GB."""
    W = weight_memory_gb(num_params_B, precision)
    KV = kv_cache_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size, precision)
    A = W * activation_pct * batch_size
    return {"weights": W, "kv_cache": KV, "activations": A, "overhead": overhead_gb,
            "total": W + KV + A + overhead_gb}

# Three scenarios
scenarios = [
    ("Llama-8B, INT4, ctx=8K, bs=1",
     dict(num_params_B=8, precision="INT4", num_layers=32, num_kv_heads=8, head_dim=128, seq_len=8192, batch_size=1)),
    ("Llama-70B, INT8, ctx=4K, bs=4",
     dict(num_params_B=70, precision="INT8", num_layers=80, num_kv_heads=8, head_dim=128, seq_len=4096, batch_size=4)),
    ("Llama-405B, FP16, ctx=2K, bs=1",
     dict(num_params_B=405, precision="FP16", num_layers=126, num_kv_heads=8, head_dim=128, seq_len=2048, batch_size=1)),
]

for name, kwargs in scenarios:
    b = vram_budget_gb(**kwargs)
    print(f"--- {name} ---")
    print(f"  Weights:     {b['weights']:6.1f} GB")
    print(f"  KV Cache:    {b['kv_cache']:6.3f} GB")
    print(f"  Activations: {b['activations']:6.2f} GB")
    print(f"  Overhead:    {b['overhead']:6.1f} GB")
    print(f"  TOTAL:       {b['total']:6.1f} GB\n")

## Exercise 4: Predict Max Batch Size

Given a GPU, how many concurrent requests can you serve?

**Free VRAM for KV = GPU_VRAM - Weights - Overhead**

**Max Batch = Free VRAM / KV_per_request**

In [ ]:
def max_batch_size(gpu_name, num_params_B, precision, num_layers, num_kv_heads,
                   head_dim, seq_len, overhead_gb=1.0, activation_pct=0.08):
    """Predict max batch size for a model on a given GPU."""
    gpu_vram = GPU_CATALOG[gpu_name]
    W = weight_memory_gb(num_params_B, precision)
    free = gpu_vram - W - overhead_gb
    if free <= 0:
        return 0  # Model doesn't even fit
    kv_per_req = kv_cache_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size=1, precision=precision)
    act_per_req = W * activation_pct
    per_req = kv_per_req + act_per_req
    return int(free / per_req) if per_req > 0 else 0

# Llama 8B INT4 on various GPUs at 8K context
model_cfg = dict(num_params_B=8, precision="INT4", num_layers=32, num_kv_heads=8, head_dim=128, seq_len=8192)

print("Max batch size for Llama-8B INT4 @ 8K context:")
print(f"{'GPU':<18} {'VRAM':>6} {'Max Batch':>10}")
print("-" * 36)
for gpu in ["T4", "A10G", "A100-40", "A100-80", "H100"]:
    bs = max_batch_size(gpu, **model_cfg)
    print(f"{gpu:<18} {GPU_CATALOG[gpu]:>4} GB {bs:>8}")

print("\n# Takeaway: batch size is the primary throughput lever once the model fits.")

## Key Takeaways

1. **Weights dominate** at small batch sizes. Quantization (INT4) gives 4x compression.
2. **KV cache dominates** at large batch sizes or long contexts. It grows as O(batch × seq_len).
3. **The budget equation** (W + KV + A + O ≤ GPU VRAM) determines your deployment constraints.
4. **Max batch size** is the free VRAM after weights divided by per-request KV + activation cost.

Next: [Lab 1.3](../01.3_batch_instance_selection/lab.ipynb) applies these budgets to choose instance types.